# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb

import os
import duckdb

import getpass

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass(
    "Paste your Hugging Face Read token: "
)

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
print("Connected to Hugging Face.")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Connected to Hugging Face.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis

One row in my March feature frame represents one pseudonymized content item for one client (`client_hash_id` × `content_hash_id`). It summarizes that item’s daily performance from 2026-03-01 through 2026-03-31.

The source fact table is more detailed: one row per `report_date` × `client_hash_id` × `content_hash_id`. I will aggregate those daily rows into one March snapshot per content item.

### Decision moment and window

The reviewer would make a prioritization decision after March ends. Therefore, all feature information must come from March. The future April outcome is used only as the label/proxy, never as a feature.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Tables

I use `fact_content_daily_performance` as the main table because it contains daily Google Search Console and GA4 performance. I may use `dim_clients` only as context for client history and analytics availability.

### Feature fields

My five planned March features are:

- Total March GSC impressions — knowable at the decision moment because impressions were recorded during March, before the April decision.
- Total March GSC clicks — knowable at the decision moment because clicks were recorded during March, before the April decision.
- March CTR — knowable at the decision moment because it is calculated only from March clicks and impressions.
- Mean March GSC average position — knowable at the decision moment because it is measured during March.
- Total March GA4 sessions — knowable at the decision moment only for observations where `ga4_data_available IS TRUE`; unavailable GA4 zeros are excluded rather than treated as no engagement.

Each is knowable at the decision moment because it was measured during March, before the reviewer makes the April prioritization decision.

### Label / proxy

I will rank content items by their observed future change in GSC clicks during April relative to March. For this notebook, a positive future-decline proxy means April GSC clicks were at least 20% below March clicks, among content items with at least 10 March clicks. This is a proxy for pages that may deserve earlier human review; it does not prove that a refresh would fix the page.

### Context fields

`client_hash_id` and `content_hash_id` identify and group rows. `report_date` defines the aggregation window. These fields are not model features.

### Excluded fields

I exclude all April measurements from the features because they occur after the March decision moment and would leak the answer. I also exclude GA4 rows where `ga4_data_available` is not true, because zero-filled GA4 values there mean “not available,” not zero engagement.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
MARCH_FACT = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/**/*.parquet'"
    f")"
)

march_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {MARCH_FACT}
""").df()

march_summary

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [5]:
grain_violations = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS rows_at_grain
    FROM {MARCH_FACT}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

grain_violations

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,rows_at_grain


No duplicate raw March records were returned, supporting the stated daily source-table grain.

In [6]:
ga4_available_summary = con.sql(f"""
    SELECT
        COUNT(*) AS rows_with_ga4_available
    FROM {MARCH_FACT}
    WHERE ga4_data_available IS TRUE
""").df()

ga4_available_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_with_ga4_available
0,413966


In [7]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS march_gsc_impressions,
        SUM(gsc_clicks) AS march_gsc_clicks,

        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)
            AS march_ctr,

        AVG(gsc_avg_position) AS march_avg_position,

        SUM(
            CASE
                WHEN ga4_data_available IS TRUE THEN ga4_sessions
            END
        ) AS march_ga4_sessions

    FROM {MARCH_FACT}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,march_gsc_impressions,march_gsc_clicks,march_ctr,march_avg_position,march_ga4_sessions
0,client_3ffa76342f366962,content_131e34d488573268,0.0,0.0,NaN,NaN,NaN
1,client_3ffa76342f366962,content_0f4a3105e62d3be8,0.0,0.0,NaN,NaN,NaN
2,client_3ffa76342f366962,content_238f41dae1122bdb,5.0,0.0,0.000000,7.20000,NaN
3,client_3ffa76342f366962,content_2197e50574bb4e84,24.0,1.0,0.041667,6.02381,4.0
4,client_3ffa76342f366962,content_3e5bf20bc99d290b,3.0,0.0,0.000000,4.50000,NaN


In [8]:
APRIL_FACT = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-04/**/*.parquet'"
    f")"
)

april_outcomes = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_gsc_clicks
    FROM {APRIL_FACT}
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

labeled_frame = feature_frame.merge(
    april_outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

# Only label pages with enough March clicks for a percentage change to be meaningful.
labeled_frame = labeled_frame[
    labeled_frame["march_gsc_clicks"] >= 10
].copy()

# Positive = clicks fell by at least 20% in the future April window.
labeled_frame["future_click_decline"] = (
    labeled_frame["april_gsc_clicks"]
    <= 0.80 * labeled_frame["march_gsc_clicks"]
).astype(int)

print("Rows in labeled frame:", len(labeled_frame))
print(labeled_frame["future_click_decline"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in labeled frame: 17926
future_click_decline
1    9304
0    8622
Name: count, dtype: int64


### Deliberate leakage experiment

I will first score the March-only features honestly. Then I will intentionally add a feature derived directly from the future label, observe the misleading score, remove it, and retain the honest result.

In [9]:
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

FEATURES = [
    "march_gsc_impressions",
    "march_gsc_clicks",
    "march_ctr",
    "march_avg_position",
    "march_ga4_sessions",
]

y = labeled_frame["future_click_decline"]

train_index, test_index = train_test_split(
    labeled_frame.index,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

honest_model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("model", DecisionTreeClassifier(max_depth=3, random_state=42)),
])

honest_model.fit(
    labeled_frame.loc[train_index, FEATURES],
    y.loc[train_index],
)

honest_predictions = honest_model.predict(
    labeled_frame.loc[test_index, FEATURES]
)
honest_probabilities = honest_model.predict_proba(
    labeled_frame.loc[test_index, FEATURES]
)[:, 1]

honest_accuracy = accuracy_score(y.loc[test_index], honest_predictions)
honest_auc = roc_auc_score(y.loc[test_index], honest_probabilities)

print(f"Honest accuracy: {honest_accuracy:.3f}")
print(f"Honest ROC-AUC:  {honest_auc:.3f}")

Honest accuracy: 0.581
Honest ROC-AUC:  0.613


In [10]:
leaky_frame = labeled_frame.copy()

# DELIBERATE LEAK: this column is the future answer itself.
leaky_frame["leaked_future_decline"] = leaky_frame["future_click_decline"]

leaky_features = FEATURES + ["leaked_future_decline"]

leaky_model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("model", DecisionTreeClassifier(max_depth=3, random_state=42)),
])

leaky_model.fit(
    leaky_frame.loc[train_index, leaky_features],
    y.loc[train_index],
)

leaky_probabilities = leaky_model.predict_proba(
    leaky_frame.loc[test_index, leaky_features]
)[:, 1]

leaky_auc = roc_auc_score(y.loc[test_index], leaky_probabilities)

print(f"Leaky ROC-AUC: {leaky_auc:.3f}")

Leaky ROC-AUC: 1.000


In [11]:
del leaky_frame["leaked_future_decline"]

assert "leaked_future_decline" not in leaky_frame.columns

final_model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("model", DecisionTreeClassifier(max_depth=3, random_state=42)),
])

final_model.fit(
    labeled_frame.loc[train_index, FEATURES],
    y.loc[train_index],
)

final_probabilities = final_model.predict_proba(
    labeled_frame.loc[test_index, FEATURES]
)[:, 1]

final_auc = roc_auc_score(y.loc[test_index], final_probabilities)

print(f"Leaky ROC-AUC: {leaky_auc:.3f}")
print(f"Honest ROC-AUC after removing leak: {final_auc:.3f}")

Leaky ROC-AUC: 1.000
Honest ROC-AUC after removing leak: 0.613


The leaky column copied `future_click_decline`, which is calculated from April outcomes. Its perfect ROC-AUC of 1.000 was therefore invalid: it gave the model the answer from the future. After removing the column, the honest March-only ROC-AUC returned to 0.613. I retain 0.613 as the valid result.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Limitation

Client histories and GA4 coverage are uneven. In the March slice, only 413,966 of 9,841,378 daily rows had `ga4_data_available IS TRUE`. Therefore, unavailable GA4 zeros cannot be interpreted as zero engagement, and GA4-based comparisons are not equally informative for every client-content item. This observational data also cannot prove that refreshing a page caused a later recovery.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.